# Sealed by Default — validation notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OWNER/REPO/blob/main/validate_colab.ipynb) [![DOI](https://zenodo.org/badge/DOI/10.5281/zenodo.23575914.svg)](https://doi.org/10.5281/zenodo.23575914)

Reproduces the simulation results of the manuscript *Sealed by Default: Privacy-Preserving Parked-Vehicle Camera Recording with Fail-Closed Sealing and Owner-or-Court Unsealing* (R. Joshi, ORCID [0009-0007-4201-7862](https://orcid.org/0009-0007-4201-7862)). Code and data: [doi:10.5281/zenodo.23575914](https://doi.org/10.5281/zenodo.23575914).

* **QUICK** (about 1 minute): ten sanity checks, then six benchmark episodes recomputed and compared value-by-value with the published per-episode results.
* **FULL** (about 1.5 h on one CPU core): every benchmark, sweep, sensitivity and mechanism run, then all 56 paper values compared with the recomputed ones (Tables IV–VI, VIII, IX; Figs. 3–5; Sections IV and VII).

Not reproducible here: Table VII (real-image check), because the photographs are personal data and are not released.

All results are **simulated**; see the paper's threats to validity.

In [ ]:
MODE = "QUICK"   # set to "FULL" to rerun everything

In [ ]:
!pip -q install numpy==2.4.4 scipy==1.17.1 pandas matplotlib
import os; os.makedirs('results', exist_ok=True)

In [ ]:
%%writefile sim.py
"""
Privacy-Adaptive Sentry -- Phase 6 simulator (CPU-only, geometric).

What this is
------------
A 2.5D geometric simulator of a parked ego vehicle with surround cameras.
Ground truth (GT) identifiability of every face and number plate is computed
from true geometry: ray-traced occlusion, camera field of view, facing angle
and pixel density, using the IEC 62676-4 DORI "identification" level.

The method under test (and every baseline) only sees the output of a separate,
noisy perception model. Primary metrics are computed on GT, never on the
perception model (principle 4).

What this is NOT
----------------
No images are rendered. Identifiability is a geometric proxy, not the output of
a re-identification network. See VALIDATION.md for assumptions and limits.
"""
from __future__ import annotations
import dataclasses as dc
import numpy as np

# --------------------------------------------------------------------------
# Parameters (every value here is an ASSUMPTION unless marked otherwise)
# --------------------------------------------------------------------------

@dc.dataclass
class Params:
    dt: float = 0.5                 # s per simulation step
    duration: float = 600.0         # s parked episode
    ped_rate_per_min: float = 6.0   # pedestrian arrivals per minute
    veh_rate_per_min: float = 2.0   # passing vehicles per minute
    incident: bool = True           # one vandal incident in the episode
    condition: str = "day_clear"
    camera_set: str = "5cam"        # "4fisheye" or "5cam" (+ front narrow)
    layout: str = "opposite_row"    # "opposite_row" or "no_opposite"
    # GT identifiability thresholds, pixels per metre at the target
    # IEC 62676-4 DORI: identification = 250 px/m (REQUIRES SOURCE VERIFICATION
    # of exact wording; value widely quoted).
    face_id_pxm: float = 250.0
    plate_id_pxm: float = 200.0     # ASSUMPTION: ~15 px character height for 75 mm characters
    face_max_angle_deg: float = 70.0   # face must point within this angle of the camera
    plate_max_angle_deg: float = 60.0
    min_visible_frac: float = 0.5
    # Event trigger (strongest realistic baseline: event-triggered clips)
    trigger_dist_m: float = 1.0
    trigger_hold_s: float = 1.0
    pre_s: float = 10.0
    post_s: float = 20.0
    # Perception model (method's internal model) -- ASSUMPTIONS, swept in Phase 7
    face_det_c50_pxm: float = 110.0   # 50 % face detection at this effective density
    face_det_scale: float = 25.0
    plate_det_c50_pxm: float = 90.0
    plate_det_scale: float = 25.0
    person_det_c50_pxm: float = 20.0
    person_det_scale: float = 8.0
    vehicle_det_c50_pxm: float = 10.0
    vehicle_det_scale: float = 5.0
    track_sigma: float = 0.6          # per-track latent offset (logit units) -> correlated misses
    frame_sigma: float = 0.4          # per-frame noise (logit units)
    occl_penalty: float = 1.5         # logit penalty per unit occluded fraction
    yaw_free_deg: float = 40.0        # detector unaffected up to this face yaw (ASSUMPTION)
    yaw_penalty_per_30deg: float = 2.0  # logit penalty per 30 deg beyond yaw_free (ASSUMPTION)
    evidence_min_s: float = 1.0       # perpetrator identifiable for >= this long
    seed: int = 0

# Image-quality factor by condition, multiplies effective pixel density.
# ASSUMPTIONS (to be replaced by measured MTF / SNR data in Phase 8).
TIME_Q = {"day": 1.00, "dusk": 0.80, "night": 0.50}        # ASSUMPTION
WEATHER_Q = {"clear": 1.00, "rain": 0.80, "fog": 0.60}      # ASSUMPTION
TIME_SHIFT = {"day": 0.0, "dusk": -0.3, "night": -0.8}      # detector logit shift, ASSUMPTION
WEATHER_SHIFT = {"clear": 0.0, "rain": -0.3, "fog": -0.6}
CONDITIONS = [f"{t}_{w}" for t in TIME_Q for w in WEATHER_Q]
# Morning and afternoon are not distinguished: the model has no sun direction or glare.

def condition_factors(cond: str):
    t, w = cond.split("_")
    return TIME_Q[t] * WEATHER_Q[w], TIME_SHIFT[t] + WEATHER_SHIFT[w]

# sim v3: fog is distance-dependent (Koschmieder). Contrast transmission T(d) = exp(-beta d),
# beta = 3.912 / V. ASSUMPTIONS: visibility V = 100 m for 'fog'; effective density scales with T(d);
# detector logit shift = -FOG_SHIFT_K * (1 - T(d)).
FOG_VISIBILITY_M = 100.0
FOG_SHIFT_K = 3.0

def condition_arrays(cond: str, d: np.ndarray):
    """Per-target (q, detector shift) for distances d (m)."""
    t, w = cond.split("_")
    if w == "fog":
        T = np.exp(-3.912 / FOG_VISIBILITY_M * d)
        return TIME_Q[t] * T, TIME_SHIFT[t] - FOG_SHIFT_K * (1 - T)
    return np.full_like(d, TIME_Q[t] * WEATHER_Q[w]), np.full_like(d, TIME_SHIFT[t] + WEATHER_SHIFT[w])

# Backwards-compatible tables (identical values to the Phase 6 smoke run)
CONDITION_Q = {c: condition_factors(c)[0] for c in CONDITIONS}
CONDITION_DET_SHIFT = {c: condition_factors(c)[1] for c in CONDITIONS}

# --------------------------------------------------------------------------
# Geometry
# --------------------------------------------------------------------------
EGO_HALF = np.array([2.35, 0.95, 0.75])   # half-extents (m); box centre z=0.75
EGO_CENTER = np.array([0.0, 0.0, 0.75])

def box_from(center, half):
    c = np.asarray(center, float); h = np.asarray(half, float)
    return c - h, c + h

@dc.dataclass
class Camera:
    name: str
    pos: np.ndarray
    axis: np.ndarray          # unit optical axis
    model: str                # "fisheye" (equidistant) or "pinhole"
    f_px: float               # focal length in px/rad (fisheye) or px (pinhole)
    half_fov_rad: float       # fisheye: max angle from axis; pinhole: half HFOV
    half_vfov_rad: float = np.pi  # pinhole only
    up: np.ndarray = dc.field(default_factory=lambda: np.array([0, 0, 1.0]))

def _unit(v):
    v = np.asarray(v, float); return v / np.linalg.norm(v)

def make_cameras(camera_set: str):
    # Fisheye: 1920 px across 190 deg, equidistant -> f = 960 / (95 deg in rad)
    f_fe = 960.0 / np.deg2rad(95.0)
    tilt = np.deg2rad(30.0)
    cams = [
        Camera("front_fe", np.array([2.35, 0.0, 0.60]), _unit([np.cos(tilt*0.3), 0, -np.sin(tilt*0.3)]), "fisheye", f_fe, np.deg2rad(95)),
        Camera("rear_fe",  np.array([-2.35, 0.0, 0.95]), _unit([-np.cos(tilt*0.3), 0, -np.sin(tilt*0.3)]), "fisheye", f_fe, np.deg2rad(95)),
        Camera("left_fe",  np.array([0.95, 1.00, 1.00]), _unit([0, np.cos(tilt*2), -np.sin(tilt*2)]), "fisheye", f_fe, np.deg2rad(95)),
        Camera("right_fe", np.array([0.95, -1.00, 1.00]), _unit([0, -np.cos(tilt*2), -np.sin(tilt*2)]), "fisheye", f_fe, np.deg2rad(95)),
    ]
    if camera_set == "5cam":
        # Front narrow camera behind windscreen: 1920 px across 60 deg HFOV
        f_n = 960.0 / np.tan(np.deg2rad(30.0))
        cams.append(Camera("front_narrow", np.array([0.6, 0.0, 1.35]), _unit([1, 0, -0.03]), "pinhole",
                           f_n, np.deg2rad(30.0), np.deg2rad(20.0)))
    return cams

def pixel_density(cam: Camera, pts: np.ndarray):
    """Return (in_fov bool, px_per_m) for points (N,3). px/m at the target
    for a small object facing the camera (fronto-parallel approximation)."""
    v = pts - cam.pos
    d = np.linalg.norm(v, axis=1)
    u = v / d[:, None]
    cosang = u @ cam.axis
    ang = np.arccos(np.clip(cosang, -1, 1))
    if cam.model == "fisheye":
        in_fov = ang < cam.half_fov_rad
        pxm = cam.f_px / d           # equidistant: angular density f px/rad
    else:
        right = _unit(np.cross(cam.axis, cam.up)); up = np.cross(right, cam.axis)
        x = u @ right; y = u @ up; z = cosang
        with np.errstate(divide="ignore", invalid="ignore"):
            az = np.arctan2(x, z); el = np.arctan2(y, z)
        in_fov = (z > 0) & (np.abs(az) < cam.half_fov_rad) & (np.abs(el) < cam.half_vfov_rad)
        # pinhole density ~ f / (d * cos(angle)) ; keep conservative f/d
        pxm = cam.f_px / d
    return in_fov, pxm, d

def ray_occluded(origins, targets, bmin, bmax, exclude):
    """Vectorised slab test. origins/targets (R,3); boxes (B,3);
    exclude (R,) index of box to ignore (-1 = none). Returns (R,) bool."""
    R = origins.shape[0]; B = bmin.shape[0]
    if B == 0:
        return np.zeros(R, bool)
    d = targets - origins
    L = np.linalg.norm(d, axis=1)
    dn = d / L[:, None]
    inv = 1.0 / np.where(np.abs(dn) < 1e-12, 1e-12, dn)
    t1 = (bmin[None] - origins[:, None]) * inv[:, None]
    t2 = (bmax[None] - origins[:, None]) * inv[:, None]
    tnear = np.max(np.minimum(t1, t2), axis=2)
    tfar = np.min(np.maximum(t1, t2), axis=2)
    hit = (tnear <= tfar) & (tfar > 1e-3) & (tnear < (L[:, None] - 0.05))
    if exclude is not None:
        idx = np.arange(B)[None, :]
        hit &= idx != exclude[:, None]
    return hit.any(axis=1)

# --------------------------------------------------------------------------
# Scene
# --------------------------------------------------------------------------
PED_HALF = np.array([0.25, 0.25, 0.875])   # body box, 1.75 m tall
VEH_HALF = np.array([2.3, 0.92, 0.75])

@dc.dataclass
class Scene:
    t: np.ndarray
    # per actor arrays over time: positions (T,A,3) NaN when absent
    ped_pos: np.ndarray
    ped_head: np.ndarray        # facing unit vector (T,A,3)
    ped_is_vandal: np.ndarray   # (A,)
    veh_pos: np.ndarray         # (T,V,3) centre
    veh_dir: np.ndarray         # (T,V,3) heading unit
    static_boxes: list          # list of (min,max) occluders (parked neighbours)
    static_plates: list         # list of (centre, normal) for neighbours' plates
    ped_head_z: np.ndarray = None  # (A,) face height above ground (m)

def build_scene(p: Params, rng: np.random.Generator) -> Scene:
    T = int(round(p.duration / p.dt)); t = np.arange(T) * p.dt
    # Parked neighbours left and right (nose-in), occlude and expose plates.
    neigh_centres = [np.array([0.0, 2.6, 0.75]), np.array([0.0, -2.6, 0.75])]
    # Opposite row across the front aisle, parked nose-in towards the ego (plates face -x)
    opp_centres = [np.array([9.5, y, 0.75]) for y in (-5.2, -2.6, 0.0, 2.6, 5.2)] if p.layout == "opposite_row" else []
    static_boxes = [box_from(c, VEH_HALF) for c in neigh_centres + opp_centres]
    static_plates = []
    for c in neigh_centres:
        static_plates.append((c + np.array([2.32, 0, -0.35]), np.array([1.0, 0, 0])))   # front plate
        static_plates.append((c + np.array([-2.32, 0, -0.25]), np.array([-1.0, 0, 0])))  # rear plate
    for c in opp_centres:
        static_plates.append((c + np.array([-2.32, 0, -0.35]), np.array([-1.0, 0, 0])))  # front plate facing ego
        static_plates.append((c + np.array([2.32, 0, -0.25]), np.array([1.0, 0, 0])))    # rear plate (away)
    # Pedestrians: sidewalk in front of the parked row (x ~ 4.5 m), plus
    # some crossing behind (x ~ -4.5 m aisle).
    n_ped = rng.poisson(p.ped_rate_per_min * p.duration / 60.0)
    A = n_ped + (1 if p.incident else 0)
    ped_pos = np.full((T, A, 3), np.nan); ped_head = np.full((T, A, 3), np.nan)
    vandal = np.zeros(A, bool)
    head_z = np.clip(rng.normal(1.62, 0.08, A), 1.45, 1.85)
    for a in range(n_ped):
        t0 = rng.uniform(0, p.duration)
        lane_x = rng.choice([4.5, -4.8]) + rng.normal(0, 0.4)
        direction = rng.choice([-1, 1])
        speed = np.clip(rng.normal(1.35, 0.2), 0.8, 2.0)
        y0 = -15.0 * direction
        # Some walkers pass close to the ego car (cut through between cars)
        close = rng.random() < 0.08
        for k in range(T):
            tau = t[k] - t0
            if tau < 0: continue
            y = y0 + direction * speed * tau
            if abs(y) > 15.5: continue
            x = lane_x
            if close and abs(y) < 3.0:
                x = 2.35 + 0.6 if lane_x > 0 else -2.35 - 0.6
            ped_pos[k, a] = [x, y, PED_HALF[2]]
            ped_head[k, a] = [0, direction, 0]
    if p.incident:
        a = A - 1; vandal[a] = True
        t_in = rng.uniform(120, p.duration - 120)
        side = rng.choice([-1, 1])
        xs = rng.uniform(-1.8, 1.8)                      # where along the car the damage happens
        ys = side * (EGO_HALF[1] + rng.uniform(0.35, 0.9))
        yaw = np.deg2rad(rng.uniform(-60, 60))           # face not always towards the car
        stand = rng.uniform(6, 20)
        come_from = rng.choice([-1, 1])                  # approach from rear (-1) or front (+1) aisle
        leave_to = rng.choice([-1, 1])
        start = np.array([come_from * 6.5, ys, PED_HALF[2]]); spot = np.array([xs, ys, PED_HALF[2]])
        end = np.array([leave_to * 7.0, ys + side * 0.3, PED_HALF[2]])
        face_car = np.array([0, -side, 0.0])
        c, s_ = np.cos(yaw), np.sin(yaw)
        stand_head = np.array([c * face_car[0] - s_ * face_car[1], s_ * face_car[0] + c * face_car[1], 0])
        head_z[a] = rng.uniform(1.5, 1.85)
        for k in range(T):
            tau = t[k] - t_in
            if -10 <= tau < 0:
                frac = (tau + 10) / 10; pos = start + (spot - start) * frac; head = _unit(spot - start)
            elif 0 <= tau < stand:
                pos = spot + np.array([0.08 * np.sin(tau), 0, 0]); head = stand_head
            elif stand <= tau < stand + 12:
                frac = (tau - stand) / 12; pos = spot + (end - spot) * frac; head = _unit(end - spot)
            else:
                continue
            ped_pos[k, a] = pos; ped_head[k, a] = head
    # Passing vehicles in the aisle behind (x ~ -6.5)
    n_veh = rng.poisson(p.veh_rate_per_min * p.duration / 60.0)
    veh_pos = np.full((T, n_veh, 3), np.nan); veh_dir = np.full((T, n_veh, 3), np.nan)
    for v in range(n_veh):
        t0 = rng.uniform(0, p.duration); direction = rng.choice([-1, 1])
        speed = rng.uniform(2.0, 5.0)
        for k in range(T):
            tau = t[k] - t0
            if tau < 0: continue
            y = -25 * direction + direction * speed * tau
            if abs(y) > 25.5: continue
            veh_pos[k, v] = [6.3 if v % 2 == 0 else -7.5, y, 0.75]; veh_dir[k, v] = [0, direction, 0]
    return Scene(t, ped_pos, ped_head, vandal, veh_pos, veh_dir, static_boxes, static_plates, head_z)

# --------------------------------------------------------------------------
# Ground truth + perception per step
# --------------------------------------------------------------------------
FACE_OFFS = np.array([[0, 0, 0], [0, 0.07, 0.06], [0, -0.07, 0.06], [0, 0.07, -0.08], [0, -0.07, -0.08]])

def observe(scene: Scene, p: Params, rng: np.random.Generator):
    """Returns dict of per-step arrays:
    face_gt[T,A]  : face identifiable by >=1 camera (GT)
    face_det[T,A] : face detected by perception in a camera where it is visible
    person_det[T,A]
    plate_gt[T,P], plate_det[T,P], vehicle_det[T,P]  (P = static + moving plates)
    near[T,A]     : distance to ego body < trigger distance
    """
    cams = make_cameras(p.camera_set)
    T, A = scene.ped_pos.shape[:2]
    V = scene.veh_pos.shape[1]
    S = len(scene.static_plates)
    P = S + 2 * V
    face_gt = np.zeros((T, A), bool); face_det = np.zeros((T, A), bool)
    person_det = np.zeros((T, A), bool); near = np.zeros((T, A), bool)
    body_vis = np.zeros((T, A), bool)   # instrumentation: body at least partly visible in some camera
    face_pxm_best = np.zeros((T, A))
    plate_gt = np.zeros((T, P), bool); plate_det = np.zeros((T, P), bool)
    vehicle_det = np.zeros((T, P), bool)
    track_face = rng.normal(0, p.track_sigma, A); track_person = rng.normal(0, p.track_sigma, A)
    track_plate = rng.normal(0, p.track_sigma, P)
    a1_tot = 0; a1_viol = 0   # instrumentation for Proposition 3, assumption A1 (no effect on results)
    cos_face = np.cos(np.deg2rad(p.face_max_angle_deg)); cos_plate = np.cos(np.deg2rad(p.plate_max_angle_deg))
    sig = lambda z: 1 / (1 + np.exp(-np.clip(z, -60, 60)))
    for k in range(T):
        present = ~np.isnan(scene.ped_pos[k, :, 0])
        pidx = np.where(present)[0]
        vpresent = ~np.isnan(scene.veh_pos[k, :, 0]); vidx = np.where(vpresent)[0]
        # occluder boxes: neighbours, moving vehicles, pedestrians
        boxes = list(scene.static_boxes)
        boxes += [box_from(scene.veh_pos[k, v], VEH_HALF) for v in vidx]
        ped_box_start = len(boxes)
        boxes += [box_from(scene.ped_pos[k, a], PED_HALF) for a in pidx]
        bmin = np.array([b[0] for b in boxes]); bmax = np.array([b[1] for b in boxes])
        # distance to ego body (2D) for trigger
        for j, a in enumerate(pidx):
            pos = scene.ped_pos[k, a]
            dx = max(abs(pos[0]) - EGO_HALF[0], 0); dy = max(abs(pos[1]) - EGO_HALF[1], 0)
            near[k, a] = np.hypot(dx, dy) < p.trigger_dist_m
        # ---- faces ----
        if len(pidx):
            hz = scene.ped_head_z[pidx] if scene.ped_head_z is not None else np.full(len(pidx), 1.6)
            heads = scene.ped_pos[k, pidx].copy(); heads[:, 2] = hz
            facing = scene.ped_head[k, pidx]
            for cam in cams:
                in_fov, pxm, d = pixel_density(cam, heads)
                q, dshift = condition_arrays(p.condition, d)
                to_cam = cam.pos - heads; to_cam /= np.linalg.norm(to_cam, axis=1)[:, None]
                cos_yaw = np.sum(to_cam * facing, axis=1)
                face_ok = cos_yaw > cos_face
                yaw_deg = np.rad2deg(np.arccos(np.clip(cos_yaw, -1, 1)))
                yaw_pen = p.yaw_penalty_per_30deg * np.maximum(yaw_deg - p.yaw_free_deg, 0) / 30.0
                # occlusion: rays to 5 face points; exclude own body box
                pts = (heads[:, None, :] + FACE_OFFS[None]).reshape(-1, 3)
                excl = np.repeat(ped_box_start + np.arange(len(pidx)), len(FACE_OFFS))
                occ = ray_occluded(np.repeat(cam.pos[None], len(pts), 0), pts, bmin, bmax, excl)
                vis_frac = 1 - occ.reshape(len(pidx), -1).mean(axis=1)
                eff = pxm * q
                gt = in_fov & face_ok & (vis_frac >= p.min_visible_frac) & (eff >= p.face_id_pxm)
                face_gt[k, pidx] |= gt
                face_pxm_best[k, pidx] = np.maximum(face_pxm_best[k, pidx], np.where(in_fov & face_ok, eff * vis_frac, 0))
                # perception (method's model): face
                z = (eff - p.face_det_c50_pxm) / p.face_det_scale + dshift + track_face[pidx] \
                    - p.occl_penalty * (1 - vis_frac) - yaw_pen + rng.normal(0, p.frame_sigma, len(pidx))
                fd = in_fov & face_ok & (vis_frac > 0) & (rng.random(len(pidx)) < sig(z))
                face_det[k, pidx] |= fd
                # person detection uses body size (~1.75 m) -> much easier
                body_pts = scene.ped_pos[k, pidx]
                in_fov_b, pxm_b, d_b = pixel_density(cam, body_pts)
                q_b, dshift_b = condition_arrays(p.condition, d_b)
                bp = np.concatenate([body_pts[:, None, :] + np.array([[0, 0, dz]]) for dz in (-0.45, 0.0, 0.6)], axis=1).reshape(-1, 3)
                occ_b3 = ray_occluded(np.repeat(cam.pos[None], len(bp), 0), bp, bmin, bmax,
                                      np.repeat(ped_box_start + np.arange(len(pidx)), 3))
                occ_b = occ_b3.reshape(len(pidx), 3).mean(axis=1)
                zb = (pxm_b * q_b - p.person_det_c50_pxm) / p.person_det_scale + dshift_b + track_person[pidx] \
                     - p.occl_penalty * occ_b + rng.normal(0, p.frame_sigma, len(pidx))
                person_det[k, pidx] |= in_fov_b & (rng.random(len(pidx)) < sig(zb))
                body_vis[k, pidx] |= in_fov_b & (occ_b < 1.0)
                a1_tot += int(gt.sum())
                a1_viol += int((gt & ((pxm_b * q_b < p.face_id_pxm) | ~in_fov_b | (occ_b > (1 - vis_frac) + 1e-9))).sum())
        # ---- plates ----
        plates = list(scene.static_plates)
        for v in vidx:
            c = scene.veh_pos[k, v]; h = scene.veh_dir[k, v]
            plates.append((c + h * 2.32 + np.array([0, 0, -0.35]), h))
            plates.append((c - h * 2.32 + np.array([0, 0, -0.25]), -h))
        plate_ids = list(range(S)) + [S + 2 * v + s for v in vidx for s in (0, 1)]
        if plates:
            pc = np.array([pl[0] for pl in plates]); pn = np.array([pl[1] for pl in plates])
            # owning vehicle box index per plate (exclude it from occlusion)
            own = np.array(list(np.repeat(np.arange(len(scene.static_boxes)), 2)) +
                           [len(scene.static_boxes) + j for j in range(len(vidx)) for _ in (0, 1)])
            pid = np.array(plate_ids)
            for cam in cams:
                in_fov, pxm, d = pixel_density(cam, pc)
                q, dshift = condition_arrays(p.condition, d)
                to_cam = cam.pos - pc; to_cam /= np.linalg.norm(to_cam, axis=1)[:, None]
                ok = np.sum(to_cam * pn, axis=1) > cos_plate
                occ = ray_occluded(np.repeat(cam.pos[None], len(pc), 0), pc + pn * 0.02, bmin, bmax, own)
                eff = pxm * q
                gt = in_fov & ok & ~occ & (eff >= p.plate_id_pxm)
                plate_gt[k, pid] |= gt
                z = (eff - p.plate_det_c50_pxm) / p.plate_det_scale + dshift + track_plate[pid] \
                    - p.occl_penalty * occ + rng.normal(0, p.frame_sigma, len(pid))
                plate_det[k, pid] |= in_fov & ok & (rng.random(len(pid)) < sig(z))
                zv = (eff * 4 - p.vehicle_det_c50_pxm) / p.vehicle_det_scale + dshift  # vehicle ~4x plate width
                vehicle_det[k, pid] |= in_fov & (rng.random(len(pid)) < sig(zv))
    return dict(face_gt=face_gt, face_det=face_det, person_det=person_det, near=near,
                face_pxm_best=face_pxm_best, plate_gt=plate_gt, plate_det=plate_det,
                vehicle_det=vehicle_det, a1_tot=a1_tot, a1_viol=a1_viol, body_vis=body_vis)

# --------------------------------------------------------------------------
# Designs (method + baselines + ablations)
# --------------------------------------------------------------------------

def event_mask(near: np.ndarray, p: Params):
    """Recording mask [T] from proximity trigger held >= trigger_hold_s."""
    T = near.shape[0]; hold = int(round(p.trigger_hold_s / p.dt))
    any_near = near.any(axis=1).astype(int)
    run = np.convolve(any_near, np.ones(hold, int), mode="full")[:T]
    trig = np.where(run >= hold)[0]
    rec = np.zeros(T, bool)
    pre = int(round(p.pre_s / p.dt)); post = int(round(p.post_s / p.dt))
    for k in trig:
        rec[max(0, k - pre): min(T, k + post + 1)] = True
    return rec

DESIGNS = {
    #  name              recording     identity handling      fail_closed  per_person_keys
    "B0_continuous_raw": ("continuous", "raw",        False, False),
    "B1_event_raw":      ("event",      "raw",        False, False),
    "B2_event_blur":     ("event",      "irreversible", False, False),
    "B2fc_event_blur_fc":("event",      "irreversible", True,  False),
    "P_proposed":        ("event",      "sealed",     True,  True),
    "A1_no_failclosed":  ("event",      "sealed",     False, True),
    "A2_continuous_sealed":("continuous","sealed",    True,  True),
    "A3_single_key":     ("event",      "sealed",     True,  False),
}

def evaluate(scene: Scene, obs: dict, p: Params, design: str):
    rec_mode, ident, fail_closed, per_person = DESIGNS[design]
    T = scene.t.size
    rec = np.ones(T, bool) if rec_mode == "continuous" else event_mask(obs["near"], p)
    fg, fd, pd = obs["face_gt"], obs["face_det"], obs["person_det"]
    plg, pld, vd = obs["plate_gt"], obs["plate_det"], obs["vehicle_det"]
    if ident == "raw":
        face_leak = fg; plate_leak = plg
    else:
        protected_face = fd | (pd if fail_closed else False)
        protected_plate = pld | (vd if fail_closed else False)
        face_leak = fg & ~protected_face; plate_leak = plg & ~protected_plate
    R = rec[:, None]
    aie_face = (face_leak & R).sum() * p.dt          # identifiable person-seconds accessible
    aie_plate = (plate_leak & R).sum() * p.dt
    captured_face = (fg & R)                          # GT identifiable within recorded time
    # Evidence utility: perpetrator identifiable (after authorised unseal) >= evidence_min_s
    vand = np.where(scene.ped_is_vandal)[0]
    ev = np.nan; collateral = 0.0; vandal_usable_s = np.nan
    if len(vand):
        a = vand[0]
        if ident in ("raw", "sealed"):
            usable = captured_face[:, a]
        else:  # irreversible: only leaked (missed) frames remain identifiable
            usable = (face_leak & R)[:, a]
        vandal_usable_s = float(usable.sum() * p.dt)
        ev = float(vandal_usable_s >= p.evidence_min_s)
        if ident == "sealed":
            # Unseal for the incident window. Per-person keys: only the vandal.
            inc = rec & scene_window(scene, a)
            others = np.ones(fg.shape[1], bool); others[a] = False
            if not per_person:
                collateral = (fg[inc][:, others]).sum() * p.dt
    total_gt_face = fg.sum() * p.dt
    return dict(design=design, aie_face_s=aie_face, aie_plate_s=aie_plate,
                aie_total_s=aie_face + aie_plate, evidence=ev, collateral_s=collateral, vandal_usable_s=vandal_usable_s,
                recorded_frac=float(rec.mean()), gt_face_s=total_gt_face)

def scene_window(scene: Scene, a: int):
    present = ~np.isnan(scene.ped_pos[:, a, 0])
    idx = np.where(present)[0]
    w = np.zeros(scene.t.size, bool)
    if len(idx): w[idx.min(): idx.max() + 1] = True
    return w

def run_episode(p: Params):
    rng = np.random.default_rng(p.seed)
    scene = build_scene(p, rng)
    obs = observe(scene, p, rng)
    return scene, obs, [evaluate(scene, obs, p, d) for d in DESIGNS]


In [ ]:
%%writefile checks.py
"""Phase 6 sanity checks. Run: python checks.py  (writes checks_result.json)"""
import json, dataclasses as dc
import numpy as np
import sim

results = []
def record(name, ok, detail):
    results.append(dict(check=name, passed=bool(ok), detail=detail))
    print(("PASS " if ok else "FAIL ") + name + " | " + detail)

# 1. Projection: point on optical axis is in FOV; point behind fisheye limit is not.
cams = {c.name: c for c in sim.make_cameras("5cam")}
c = cams["front_narrow"]
on_axis = c.pos + c.axis * 10
inside, pxm, d = sim.pixel_density(c, on_axis[None])
behind = c.pos - c.axis * 10
inside_b, _, _ = sim.pixel_density(c, behind[None])
record("projection_on_axis", inside[0] and not inside_b[0] and abs(pxm[0] - c.f_px / 10) < 1e-6,
       f"in_fov={inside[0]}, behind_in_fov={inside_b[0]}, px/m={pxm[0]:.1f} (expected {c.f_px/10:.1f})")
fe = cams["front_fe"]
ang_ok = []
for deg in (90, 94, 96, 120):
    th = np.deg2rad(deg)
    # rotate axis in x-y plane by deg (axis is nearly +x)
    v = np.array([np.cos(th), np.sin(th), fe.axis[2]]); v /= np.linalg.norm(v)
    ins, _, _ = sim.pixel_density(fe, (fe.pos + v * 5)[None])
    ang_ok.append(ins[0])
record("fisheye_fov_boundary", ang_ok[0] and not ang_ok[3],
       f"in_fov at ~90/94/96/120 deg = {ang_ok} (95 deg half-FOV; small tilt shifts boundary)")

# 2. Known-answer identification distance for the narrow camera (day, no occlusion).
p = sim.Params()
d_expected = c.f_px / p.face_id_pxm
record("identification_distance_analytic", True,
       f"front_narrow identification range = {d_expected:.2f} m; fisheye = {cams['front_fe'].f_px / p.face_id_pxm:.2f} m (f/250)")

# 3. Occlusion: ray through a box is occluded; ray beside it is not.
bmin, bmax = sim.box_from([5, 0, 0.75], [1, 1, 0.75])
o = np.array([[0, 0, 1.0], [0, 3, 1.0]]); t = np.array([[10, 0, 1.0], [10, 3, 1.0]])
occ = sim.ray_occluded(o, t, bmin[None], bmax[None], np.array([-1, -1]))
record("ray_occlusion_known_answer", occ[0] and not occ[1], f"through={occ[0]}, beside={occ[1]}")

# 4. Single pedestrian walking straight at the front narrow camera: GT onset distance.
p1 = sim.Params(duration=30, ped_rate_per_min=0, veh_rate_per_min=0, incident=False, seed=1)
T = int(p1.duration / p1.dt)
scene = sim.Scene(t=np.arange(T) * p1.dt, ped_pos=np.full((T, 1, 3), np.nan), ped_head=np.full((T, 1, 3), np.nan),
                  ped_is_vandal=np.zeros(1, bool), veh_pos=np.zeros((T, 0, 3)), veh_dir=np.zeros((T, 0, 3)),
                  static_boxes=[], static_plates=[])
xs = np.linspace(25, 3, T)
for k in range(T):
    scene.ped_pos[k, 0] = [xs[k], 0, sim.PED_HALF[2]]; scene.ped_head[k, 0] = [-1, 0, 0]
obs = sim.observe(scene, p1, np.random.default_rng(1))
first = np.argmax(obs["face_gt"][:, 0])
head = np.array([xs[first], 0, 1.6])
d_first = np.linalg.norm(head - c.pos)
record("single_pedestrian_onset", abs(d_first - d_expected) < 0.6,
       f"GT identifiable from {d_first:.2f} m (analytic {d_expected:.2f} m, step 0.5 s at ~0.37 m/step)")

# 5. Pedestrian fully hidden behind a parked car: never identifiable.
scene2 = dc.replace(scene, static_boxes=[sim.box_from([1.5, 0, 0.9], [0.3, 3.0, 0.9])])
scene2.ped_pos = np.full((T, 1, 3), np.nan); scene2.ped_head = np.full((T, 1, 3), np.nan)
for k in range(T):
    scene2.ped_pos[k, 0] = [6.0, 0, sim.PED_HALF[2]]; scene2.ped_head[k, 0] = [-1, 0, 0]
# camera set without narrow: use fisheye only; wall 1.8 m high covers face at 1.6 m from low cameras
obs2 = sim.observe(scene2, dc.replace(p1, camera_set="4fisheye"), np.random.default_rng(2))
record("full_occlusion_zero", obs2["face_gt"].sum() == 0, f"identifiable steps = {int(obs2['face_gt'].sum())}")

# 6. Perfect detector -> sealed/blurred designs leak nothing.
pp = sim.Params(seed=3, face_det_c50_pxm=-1e6, plate_det_c50_pxm=-1e6, person_det_c50_pxm=-1e6, vehicle_det_c50_pxm=-1e6)
scene3, obs3, res3 = sim.run_episode(pp)
leak = {r["design"]: r["aie_total_s"] for r in res3}
record("perfect_detector_no_leak", leak["P_proposed"] == 0 and leak["B2_event_blur"] == 0 and leak["A1_no_failclosed"] == 0,
       f"AIE P={leak['P_proposed']}, B2={leak['B2_event_blur']}, A1={leak['A1_no_failclosed']}, B1(raw)={leak['B1_event_raw']}")

# 7. Blind detector without fail-closed -> sealed design equals raw baseline (wrapper check).
pb = sim.Params(seed=3, face_det_c50_pxm=1e6, plate_det_c50_pxm=1e6, person_det_c50_pxm=1e6, vehicle_det_c50_pxm=1e6)
_, _, resb = sim.run_episode(pb)
lb = {r["design"]: r["aie_total_s"] for r in resb}
record("blind_detector_equals_raw", lb["A1_no_failclosed"] == lb["B1_event_raw"] and lb["P_proposed"] == lb["B1_event_raw"],
       f"A1={lb['A1_no_failclosed']}, P={lb['P_proposed']}, B1={lb['B1_event_raw']}")

# 8. Determinism: same seed -> identical results; different seed -> different scene.
_, _, ra = sim.run_episode(sim.Params(seed=7)); _, _, rb = sim.run_episode(sim.Params(seed=7))
_, _, rc = sim.run_episode(sim.Params(seed=8))
same = all(abs(x["aie_total_s"] - y["aie_total_s"]) < 1e-12 for x, y in zip(ra, rb))
diff = any(abs(x["aie_total_s"] - y["aie_total_s"]) > 0 for x, y in zip(ra, rc))
record("seed_determinism", same and diff, f"same-seed identical={same}, other-seed differs={diff}")

# 9. Metric independence: GT does not change when the perception model changes.
s1, o1, _ = sim.run_episode(sim.Params(seed=11))
s2, o2, _ = sim.run_episode(sim.Params(seed=11, face_det_c50_pxm=300, person_det_c50_pxm=200))
record("gt_independent_of_perception", np.array_equal(o1["face_gt"], o2["face_gt"]) and np.array_equal(o1["plate_gt"], o2["plate_gt"]),
       "face_gt and plate_gt identical under a different perception model (same seed)")

json.dump(results, open("checks_result.json", "w"), indent=2)
print(f"\n{sum(r['passed'] for r in results)}/{len(results)} checks passed")


In [ ]:
%%writefile bench.py
"""Phase 7 benchmark. Usage: python bench.py main|sweep
Writes one CSV row per (episode, design) to results/<mode>_episodes.csv (appends; resumable)."""
import csv, os, sys, time, itertools, dataclasses as dc
import sim

MODE = sys.argv[1]
BUDGET = float(sys.argv[2]) if len(sys.argv) > 2 else 1e9
SEEDS = range(15)
if MODE == "main":
    grid = [dict(condition=c, ped_rate_per_min=d, layout=l, seed=s)
            for c in sim.CONDITIONS for d in (2.0, 6.0, 15.0) for l in ("opposite_row", "no_opposite") for s in SEEDS]
elif MODE == "sweep":
    grid = [dict(condition=c, face_det_c50_pxm=f, plate_det_c50_pxm=f * 90 / 110, track_sigma=ts, seed=s)
            for c in ("day_clear", "night_clear") for f in (110.0, 180.0, 250.0, 320.0) for ts in (0.6, 1.5) for s in SEEDS]
elif MODE == "sweep_person":
    # Stress the fail-closed fallback: weaker person/vehicle detection
    grid = [dict(condition=c, person_det_c50_pxm=f, vehicle_det_c50_pxm=f / 2, seed=s)
            for c in ("day_clear", "night_clear") for f in (20.0, 120.0, 250.0, 400.0) for s in SEEDS]
else:
    raise SystemExit("mode must be main, sweep or sweep_person")

out = f"results/{MODE}_episodes.csv"
done = set()
if os.path.exists(out):
    with open(out) as fh:
        for r in csv.DictReader(fh):
            done.add(r["episode_key"])
fields = None; t0 = time.time(); n_new = 0
for g in grid:
    # seed offset per scenario so scenarios are not copies of each other, but identical across designs
    key = "|".join(f"{k}={v}" for k, v in sorted(g.items()))
    if key in done: continue
    if time.time() - t0 > BUDGET:
        print(f"BUDGET STOP {MODE}: {n_new} new episodes", flush=True); break
    p = sim.Params(**{**g, "seed": 1000 * g["seed"] + 7})
    _, _, res = sim.run_episode(p)
    rows = []
    for r in res:
        row = dict(episode_key=key, **{k: v for k, v in g.items()}, **r)
        rows.append(row)
    new_file = not os.path.exists(out)
    with open(out, "a", newline="") as fh:
        w = csv.DictWriter(fh, fieldnames=list(rows[0].keys()))
        if new_file: w.writeheader()
        w.writerows(rows)
    n_new += 1
    if n_new % 25 == 0:
        print(f"{n_new} new episodes, {time.time()-t0:.0f}s", flush=True)
print(f"DONE {MODE}: {n_new} new episodes in {time.time()-t0:.0f}s", flush=True)


In [ ]:
%%writefile analyze.py
"""Phase 7 analysis: bootstrap CIs, paired tests with Holm correction, effect sizes, figures.
Usage: python analyze.py   (reads results/main_episodes.csv and results/sweep_episodes.csv)"""
import json, numpy as np, pandas as pd
from scipy import stats
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

RNG = np.random.default_rng(12345)
B = 2000
main = pd.read_csv("results/main_episodes.csv")
sweep = pd.read_csv("results/sweep_episodes.csv")
sweep_p = pd.read_csv("results/sweep_person_episodes.csv")
for df in (main, sweep, sweep_p):
    df["aie_total_s"] = df["aie_face_s"] + df["aie_plate_s"]

def boot_mean(x):
    x = np.asarray(x, float); x = x[~np.isnan(x)]
    if len(x) == 0: return (np.nan, np.nan, np.nan)
    idx = RNG.integers(0, len(x), (B, len(x)))
    m = x[idx].mean(axis=1)
    return (x.mean(), np.percentile(m, 2.5), np.percentile(m, 97.5))

def wide(df, metric):
    return df.pivot_table(index="episode_key", columns="design", values=metric)

def wilcoxon_paired(a, b):
    d = np.asarray(a) - np.asarray(b); d = d[~np.isnan(d)]
    nz = d[d != 0]
    if len(nz) == 0:
        return dict(n=len(d), n_nonzero=0, p=1.0, r_rb=0.0, mean_diff=0.0, ci=(0.0, 0.0), note="all paired differences are zero")
    res = stats.wilcoxon(nz)
    ranks = stats.rankdata(np.abs(nz))
    wpos = ranks[nz > 0].sum(); wneg = ranks[nz < 0].sum()
    r_rb = (wpos - wneg) / (wpos + wneg)
    idx = RNG.integers(0, len(d), (B, len(d)))
    md = d[idx].mean(axis=1)
    return dict(n=len(d), n_nonzero=len(nz), p=float(res.pvalue), r_rb=float(r_rb), mean_diff=float(d.mean()),
                ci=(float(np.percentile(md, 2.5)), float(np.percentile(md, 97.5))), note="")

def mcnemar(a, b):
    a = np.asarray(a); b = np.asarray(b); m = ~(np.isnan(a) | np.isnan(b)); a = a[m]; b = b[m]
    n10 = int(((a == 1) & (b == 0)).sum()); n01 = int(((a == 0) & (b == 1)).sum())
    p = 1.0 if n10 + n01 == 0 else stats.binomtest(n10, n10 + n01, 0.5).pvalue
    d = a - b; idx = RNG.integers(0, len(d), (B, len(d))); md = d[idx].mean(axis=1)
    return dict(n=len(d), n10=n10, n01=n01, p=float(p), diff=float(d.mean()),
                ci=(float(np.percentile(md, 2.5)), float(np.percentile(md, 97.5))))

def holm(pvals):
    order = np.argsort(pvals); m = len(pvals); adj = np.empty(m); running = 0
    for rank, i in enumerate(order):
        running = max(running, min(1.0, (m - rank) * pvals[i])); adj[i] = running
    return adj

out = {}
# ---------------- primary tests (all main episodes) ----------------
A = wide(main, "aie_total_s"); E = wide(main, "evidence"); C = wide(main, "collateral_s")
tests = [
    ("T1 AIE: proposed vs event-raw (B1)", "wilcoxon", A["P_proposed"], A["B1_event_raw"]),
    ("T2 Evidence: proposed vs fail-closed blur (B2fc)", "mcnemar", E["P_proposed"], E["B2fc_event_blur_fc"]),
    ("T3 AIE: proposed vs fail-closed blur (B2fc)", "wilcoxon", A["P_proposed"], A["B2fc_event_blur_fc"]),
    ("T4 Evidence: proposed vs event-raw (B1)", "mcnemar", E["P_proposed"], E["B1_event_raw"]),
    ("T5 AIE: ablation no fail-closed (A1) vs proposed", "wilcoxon", A["A1_no_failclosed"], A["P_proposed"]),
    ("T6 Collateral: single key (A3) vs proposed", "wilcoxon", C["A3_single_key"], C["P_proposed"]),
]
rows = []
for name, kind, a, b in tests:
    r = wilcoxon_paired(a, b) if kind == "wilcoxon" else mcnemar(a, b)
    r.update(test=name, kind=kind); rows.append(r)
adj = holm(np.array([r["p"] for r in rows]))
for r, pa in zip(rows, adj): r["p_holm"] = float(pa)
out["primary_tests"] = rows

# ---------------- means with CIs by design ----------------
designs = ["B0_continuous_raw", "B1_event_raw", "B2_event_blur", "B2fc_event_blur_fc", "P_proposed",
           "A1_no_failclosed", "A2_continuous_sealed", "A3_single_key"]
overall = []
for d in designs:
    sub = main[main.design == d]
    overall.append(dict(design=d, aie=boot_mean(sub.aie_total_s), aie_face=boot_mean(sub.aie_face_s),
                        aie_plate=boot_mean(sub.aie_plate_s), evidence=boot_mean(sub.evidence),
                        collateral=boot_mean(sub.collateral_s), recorded=float(sub.recorded_frac.mean())))
out["overall"] = overall

# by condition and by density (selected designs)
sel = ["B1_event_raw", "B2fc_event_blur_fc", "P_proposed", "A1_no_failclosed"]
bycond = {}
for c in main.condition.unique():
    for d in sel:
        sub = main[(main.condition == c) & (main.design == d)]
        bycond[f"{c}|{d}"] = dict(aie=boot_mean(sub.aie_total_s), evidence=boot_mean(sub.evidence))
out["by_condition"] = bycond
bydens = {}
for dens in sorted(main.ped_rate_per_min.unique()):
    for d in sel + ["A3_single_key"]:
        sub = main[(main.ped_rate_per_min == dens) & (main.design == d)]
        bydens[f"{dens}|{d}"] = dict(aie_face=boot_mean(sub.aie_face_s), aie=boot_mean(sub.aie_total_s),
                                    evidence=boot_mean(sub.evidence), collateral=boot_mean(sub.collateral_s))
out["by_density"] = bydens
bylay = {}
for lay in main.layout.unique():
    for d in sel:
        sub = main[(main.layout == lay) & (main.design == d)]
        bylay[f"{lay}|{d}"] = dict(aie_face=boot_mean(sub.aie_face_s), aie_plate=boot_mean(sub.aie_plate_s))
out["by_layout"] = bylay

# ---------------- sweep ----------------
sw = {}
for (c, f, ts), g in sweep.groupby(["condition", "face_det_c50_pxm", "track_sigma"]):
    for d in ["B1_event_raw", "B2fc_event_blur_fc", "P_proposed", "A1_no_failclosed"]:
        sub = g[g.design == d]
        sw[f"{c}|{f}|{ts}|{d}"] = dict(aie_face=boot_mean(sub.aie_face_s), aie=boot_mean(sub.aie_total_s),
                                      evidence=boot_mean(sub.evidence))
out["sweep"] = sw
swp = {}
for (c, f), g in sweep_p.groupby(["condition", "person_det_c50_pxm"]):
    for d in ["B1_event_raw", "B2fc_event_blur_fc", "P_proposed", "A1_no_failclosed"]:
        sub = g[g.design == d]
        swp[f"{c}|{f}|{d}"] = dict(aie=boot_mean(sub.aie_total_s), aie_face=boot_mean(sub.aie_face_s), evidence=boot_mean(sub.evidence))
out["sweep_person"] = swp
# v1-flaw check: evidence must vary within conditions
ev_var = {}
for c, g in main[main.design == "P_proposed"].groupby("condition"):
    ev_var[c] = dict(mean=float(g.evidence.mean()), n=int(len(g)), n_pos=int(g.evidence.sum()))
out["evidence_within_condition"] = ev_var
out["n_sweep_person_episodes"] = int(sweep_p.episode_key.nunique())
out["n_main_episodes"] = int(main.episode_key.nunique()); out["n_sweep_episodes"] = int(sweep.episode_key.nunique())
json.dump(out, open("results/summary.json", "w"), indent=1, default=float)

# ---------------- figures ----------------
conds = [f"{t}_{w}" for t in ("day", "dusk", "night") for w in ("clear", "rain", "fog")]
labels = {"B1_event_raw": "Event raw (B1)", "B2fc_event_blur_fc": "Blur, fail-closed (B2fc)",
          "P_proposed": "Proposed", "A1_no_failclosed": "Proposed w/o fail-closed (A1)"}
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
for ax, metric, title in [(axes[0], "evidence", "Evidence utility (share of incidents)"),
                          (axes[1], "aie", "Accessible identifiable exposure (s / 10 min)")]:
    M = np.array([[bycond[f"{c}|{d}"][metric][0] for d in sel] for c in conds])
    im = ax.imshow(M, cmap="Blues", aspect="auto")
    ax.set_xticks(range(len(sel))); ax.set_xticklabels([labels[d] for d in sel], rotation=20, ha="right", fontsize=8)
    ax.set_yticks(range(len(conds))); ax.set_yticklabels([c.replace("_", ", ") for c in conds], fontsize=8)
    for i in range(M.shape[0]):
        for j in range(M.shape[1]):
            ax.text(j, i, f"{M[i,j]:.2f}" if metric == "evidence" else f"{M[i,j]:.0f}", ha="center", va="center",
                    fontsize=7, color="white" if M[i, j] > M.max() * 0.6 else "black")
    ax.set_title(title, fontsize=10)
fig.suptitle("Simulated, geometric ground truth; detector and condition factors are assumptions", fontsize=8, y=0.02)
fig.tight_layout(rect=(0, 0.04, 1, 1)); fig.savefig("results/fig_condition_heatmap.png", dpi=150)

fig, ax = plt.subplots(1, 2, figsize=(10, 3.8))
dens = sorted(main.ped_rate_per_min.unique())
for d in sel:
    y = [bydens[f"{x}|{d}"]["aie_face"] for x in dens]
    ax[0].errorbar(dens, [v[0] for v in y], yerr=[[v[0]-v[1] for v in y], [v[2]-v[0] for v in y]], marker="o", capsize=3, label=labels[d])
    y = [bydens[f"{x}|{d}"]["evidence"] for x in dens]
    ax[1].errorbar(dens, [v[0] for v in y], yerr=[[v[0]-v[1] for v in y], [v[2]-v[0] for v in y]], marker="o", capsize=3, label=labels[d])
ax[0].set_xlabel("Pedestrians per minute"); ax[0].set_ylabel("Face AIE (s / 10 min)")
ax[1].set_xlabel("Pedestrians per minute"); ax[1].set_ylabel("Evidence utility")
ax[0].legend(fontsize=7); fig.tight_layout(); fig.savefig("results/fig_density.png", dpi=150)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.8), sharey=True)
for ax, c in zip(axes, ["day_clear", "night_clear"]):
    for d, ls in [("B1_event_raw", "-"), ("B2fc_event_blur_fc", "--"), ("P_proposed", "-"), ("A1_no_failclosed", ":")]:
        for ts, mk in [(0.6, "o"), (1.5, "s")]:
            xs = sorted(sweep.face_det_c50_pxm.unique())
            ys = [sw[f"{c}|{x}|{ts}|{d}"]["aie_face"][0] for x in xs]
            ax.plot(xs, ys, ls, marker=mk, label=f"{labels[d]}, σ={ts}")
    ax.axvline(250, color="grey", lw=0.8); ax.text(252, ax.get_ylim()[1]*0.9 if ax.get_ylim()[1] > 0 else 1, "ID threshold", fontsize=7)
    ax.set_title(c.replace("_", ", "), fontsize=10); ax.set_xlabel("Detector 50 % point (px/m)")
axes[0].set_ylabel("Face AIE (s / 10 min)"); axes[1].legend(fontsize=6)
fig.tight_layout(); fig.savefig("results/fig_detector_sweep.png", dpi=150)
print("analysis written: results/summary.json + 3 figures")


In [ ]:
%%writefile joint_stress.py
"""Joint stress: weak face/plate detector (c50=320 px/m) AND weakening person/vehicle detector."""
import csv, numpy as np, sim
rows = []
for c in ("day_clear", "night_clear"):
    for f in (20.0, 120.0, 250.0, 400.0):
        for s in range(10):
            p = sim.Params(condition=c, face_det_c50_pxm=320.0, plate_det_c50_pxm=320.0 * 90 / 110,
                           person_det_c50_pxm=f, vehicle_det_c50_pxm=f / 2, seed=1000 * s + 7)
            _, _, res = sim.run_episode(p)
            for r in res:
                if r["design"] in ("B1_event_raw", "P_proposed", "A1_no_failclosed", "B2fc_event_blur_fc"):
                    rows.append(dict(condition=c, person_det_c50_pxm=f, seed=s, **r))
with open("results/sweep_joint_episodes.csv", "w", newline="") as fh:
    w = csv.DictWriter(fh, fieldnames=list(rows[0].keys())); w.writeheader(); w.writerows(rows)
import pandas as pd
d = pd.DataFrame(rows)
print(d.groupby(["condition", "person_det_c50_pxm", "design"])[["aie_face_s", "aie_total_s", "evidence"]].mean().round(2).to_string())


In [ ]:
%%writefile batch2.py
"""Phase 11 batch 2 sensitivity runs (resumable). Usage: python batch2.py BUDGET_S
Writes results/batch2_rows.csv (one row per experiment variant x episode x design) and results/batch2_a1.csv."""
import csv, os, sys, time, dataclasses as dc, numpy as np, sim
BUDGET = float(sys.argv[1]) if len(sys.argv) > 1 else 1e9
OUT, OUT_A1 = "results/batch2_rows.csv", "results/batch2_a1.csv"
done = set()
if os.path.exists(OUT):
    with open(OUT) as fh: done = {r["job"] for r in csv.DictReader(fh)}
def near_mask(scene, dist):
    x = np.abs(scene.ped_pos[:, :, 0]); y = np.abs(scene.ped_pos[:, :, 1])
    dx = np.clip(x - sim.EGO_HALF[0], 0, None); dy = np.clip(y - sim.EGO_HALF[1], 0, None)
    d = np.hypot(dx, dy); return np.nan_to_num(d, nan=1e9) < dist
def write(path, rows):
    new = not os.path.exists(path)
    with open(path, "a", newline="") as fh:
        w = csv.DictWriter(fh, fieldnames=list(rows[0].keys()))
        if new: w.writeheader()
        w.writerows(rows)
jobs = []
for c in sim.CONDITIONS:
    for s in range(15): jobs.append(("base", c, s, None))
for v in (0.35, 0.5, 0.65):
    for s in range(15): jobs.append(("night_q", "night_clear", s, v))
for v in (0.7, 0.8, 0.9):
    for s in range(15): jobs.append(("rain_q", "day_rain", s, v))
TRIG = [(5, 10, 1.0), (10, 20, 1.0), (20, 40, 1.0), (10, 20, 0.5), (10, 20, 1.5)]
t0 = time.time(); n = 0
for exp, cond, s, v in jobs:
    job = f"{exp}|{cond}|{s}|{v}"
    if job in done: continue
    if time.time() - t0 > BUDGET: print("BUDGET STOP", n); break
    tq, wq = dict(sim.TIME_Q), dict(sim.WEATHER_Q)
    if exp == "night_q": sim.TIME_Q["night"] = v
    if exp == "rain_q": sim.WEATHER_Q["rain"] = v
    try:
        p = sim.Params(condition=cond, ped_rate_per_min=6.0, layout="opposite_row", seed=1000 * s + 7)
        rng = np.random.default_rng(p.seed); scene = sim.build_scene(p, rng); obs = sim.observe(scene, p, rng)
    finally:
        sim.TIME_Q.update(tq); sim.WEATHER_Q.update(wq)
    rows = []
    variants = [("trig", f"{a}/{b}/{d}", dc.replace(p, pre_s=a, post_s=b, trigger_dist_m=d)) for a, b, d in TRIG] if exp == "base" else [(exp, str(v), p)]
    for ename, vname, pv in variants:
        o2 = dict(obs); o2["near"] = near_mask(scene, pv.trigger_dist_m)
        for d in ("B1_event_raw", "B2fc_event_blur_fc", "P_proposed", "A3_single_key"):
            r = sim.evaluate(scene, o2, pv, d)
            rows.append(dict(job=job, exp=ename, variant=vname, condition=cond, seed=s, design=d,
                             aie_total_s=r["aie_total_s"], aie_face_s=r["aie_face_s"], evidence=r["evidence"],
                             vandal_usable_s=r["vandal_usable_s"], collateral_s=r["collateral_s"], recorded_frac=r["recorded_frac"]))
    write(OUT, rows)
    if exp == "base": write(OUT_A1, [dict(condition=cond, seed=s, a1_tot=obs["a1_tot"], a1_viol=obs["a1_viol"])])
    n += 1
print("DONE", n, "jobs in", round(time.time() - t0), "s")


In [ ]:
%%writefile analyze_batch2.py
"""Phase 11 batch 2 analysis: cluster bootstrap (R3), evidence-threshold (R7), trigger (R6), condition factors (R16), A1 (R18)."""
import json, numpy as np, pandas as pd
from scipy import stats
rng = np.random.default_rng(2026); B = 4000
m = pd.read_csv("results/main_episodes.csv"); m["seedc"] = m["seed"]
out = {}
def wide(metric): return m.pivot_table(index=["seedc", "episode_key"], columns="design", values=metric)
def cluster_ci(diff_series):
    g = diff_series.groupby(level=0); seeds = np.array(list(g.groups.keys()))
    sums = g.sum().values; cnt = g.count().values
    est = sums.sum() / cnt.sum(); bs = []
    for _ in range(B):
        idx = rng.integers(0, len(seeds), len(seeds)); bs.append(sums[idx].sum() / cnt[idx].sum())
    seed_means = sums / cnt
    w = stats.wilcoxon(seed_means) if np.any(seed_means != 0) else None
    return dict(est=float(est), ci=[float(np.percentile(bs, 2.5)), float(np.percentile(bs, 97.5))],
                seed_level_p=float(w.pvalue) if w else 1.0, n_clusters=int(len(seeds)))
A, E, C = wide("aie_total_s"), wide("evidence"), wide("collateral_s")
out["cluster"] = {"T1_AIE_P_minus_B1": cluster_ci(A["P_proposed"] - A["B1_event_raw"]),
                  "T2_Ev_P_minus_B2fc": cluster_ci(E["P_proposed"] - E["B2fc_event_blur_fc"]),
                  "T6_Coll_A3_minus_P": cluster_ci(C["A3_single_key"] - C["P_proposed"]),
                  "B1_AIE_mean": cluster_ci(A["B1_event_raw"]), "P_evidence_mean": cluster_ci(E["P_proposed"])}
ps = [out["cluster"][k]["seed_level_p"] for k in ("T1_AIE_P_minus_B1", "T2_Ev_P_minus_B2fc", "T6_Coll_A3_minus_P")]
order = np.argsort(ps); adj = [0]*3; run = 0
for r, i in enumerate(order): run = max(run, min(1, (3 - r) * ps[i])); adj[i] = run
for k, a in zip(("T1_AIE_P_minus_B1", "T2_Ev_P_minus_B2fc", "T6_Coll_A3_minus_P"), adj): out["cluster"][k]["seed_level_p_holm"] = float(a)
b = pd.read_csv("results/batch2_rows.csv")
base = b[(b.exp == "trig") & (b.variant == "10/20/1.0")]
tau = {}
for t in (0.5, 1.0, 2.0, 3.0):
    tau[str(t)] = {d: float((base[base.design == d].vandal_usable_s >= t).mean()) for d in ("B1_event_raw", "B2fc_event_blur_fc", "P_proposed")}
out["evidence_threshold"] = tau
trig = {}
for v, g in b[b.exp == "trig"].groupby("variant"):
    trig[v] = dict(B1_aie=float(g[g.design == "B1_event_raw"].aie_total_s.mean()), P_aie=float(g[g.design == "P_proposed"].aie_total_s.mean()),
                   P_evidence=float(g[g.design == "P_proposed"].evidence.mean()), B2fc_evidence=float(g[g.design == "B2fc_event_blur_fc"].evidence.mean()),
                   A3_collateral=float(g[g.design == "A3_single_key"].collateral_s.mean()), recorded=float(g.recorded_frac.mean()))
out["trigger"] = trig
cf = {}
for (e, v), g in b[b.exp.isin(["night_q", "rain_q"])].groupby(["exp", "variant"]):
    cf[f"{e}={v}"] = dict(B1_aie=float(g[g.design == "B1_event_raw"].aie_total_s.mean()), P_aie=float(g[g.design == "P_proposed"].aie_total_s.mean()),
                          P_evidence=float(g[g.design == "P_proposed"].evidence.mean()))
out["condition_factor"] = cf
a1 = pd.read_csv("results/batch2_a1.csv")
out["A1"] = dict(overall=float(a1.a1_viol.sum() / max(1, a1.a1_tot.sum())), total_face_camera_steps=int(a1.a1_tot.sum()),
                 by_condition={c: float(g.a1_viol.sum() / max(1, g.a1_tot.sum())) for c, g in a1.groupby("condition")})
json.dump(out, open("results/batch2_summary.json", "w"), indent=1)
print(json.dumps(out, indent=1))


In [ ]:
%%writefile batch3.py
"""Phase 11 batch 3: owner-view utility (R2), tracking ID swaps (R8), dropped frames (R10). Resumable.
Usage: python batch3.py BUDGET_S  -> results/batch3_rows.csv"""
import csv, os, sys, time, numpy as np, sim
BUDGET = float(sys.argv[1]) if len(sys.argv) > 1 else 1e9
OUT = "results/batch3_rows.csv"
done = set()
if os.path.exists(OUT):
    with open(OUT) as fh: done = {r["job"] for r in csv.DictReader(fh)}
SWAP = [0.0, 0.01, 0.05, 0.1, 0.2]       # ID-swap probability per step while another person is within 2 m of the vandal (ASSUMPTION)
DROP = [0.0, 0.05, 0.2, 0.5]             # frame-deadline miss probability per step (ASSUMPTION)
t0 = time.time(); n = 0
for cond in sim.CONDITIONS:
    for s in range(15):
        job = f"{cond}|{s}"
        if job in done: continue
        if time.time() - t0 > BUDGET: print("BUDGET STOP", n); break
        p = sim.Params(condition=cond, ped_rate_per_min=6.0, layout="opposite_row", seed=1000 * s + 7)
        rng = np.random.default_rng(p.seed); scene = sim.build_scene(p, rng); obs = sim.observe(scene, p, rng)
        rec = sim.event_mask(obs["near"], p)
        a = int(np.where(scene.ped_is_vandal)[0][0])
        fg, fd, pd, bv = obs["face_gt"], obs["face_det"], obs["person_det"], obs["body_vis"]
        present = ~np.isnan(scene.ped_pos[:, a, 0])
        inc = present & rec & bv[:, a]                       # incident frames the raw owner view shows
        whole = pd[:, a] & ~fd[:, a]                         # fail-closed seals the whole person
        rows = []
        base = dict(job=job, condition=cond, seed=s, n_incident_frames=int(inc.sum()))
        if inc.sum():
            rows.append(dict(base, metric="owner_view", variant="raw", value=1.0))
            rows.append(dict(base, metric="owner_view", variant="seal_whole_person", value=float((inc & ~whole).sum() / inc.sum())))
            rows.append(dict(base, metric="owner_view", variant="seal_head_only", value=1.0))
        # tracking ID swaps (Monte Carlo, fixed seed per episode)
        mrng = np.random.default_rng(p.seed + 99)
        win = np.where(present)[0]; w0, w1 = win.min(), win.max() + 1
        others = [j for j in range(fg.shape[1]) if j != a]
        for sw in SWAP:
            coll = []; ev = []
            for rep in range(20):
                swapped = np.zeros(fg.shape[1], bool); lost_from = None; c = 0.0
                for k in range(w0, w1):
                    for j in others:
                        if np.isnan(scene.ped_pos[k, j, 0]): continue
                        if not swapped[j] and np.hypot(*(scene.ped_pos[k, j, :2] - scene.ped_pos[k, a, :2])) < 2.0 and mrng.random() < sw:
                            swapped[j] = True
                            if lost_from is None: lost_from = k
                        if swapped[j] and rec[k] and fg[k, j]: c += p.dt
                usable = fg[w0:w1, a] & rec[w0:w1]
                if lost_from is not None: usable[lost_from - w0:] = False
                coll.append(c); ev.append(float(usable.sum() * p.dt >= p.evidence_min_s))
            rows.append(dict(base, metric="swap_collateral", variant=sw, value=float(np.mean(coll))))
            rows.append(dict(base, metric="swap_evidence", variant=sw, value=float(np.mean(ev))))
        # dropped frames: seal-whole-frame keeps evidence; dropping loses it
        drng = np.random.default_rng(p.seed + 7)
        for dr in DROP:
            evk = []; evd = []; ovl = []
            for rep in range(20):
                dropped = drng.random(fg.shape[0]) < dr
                u = fg[:, a] & rec
                evk.append(float(u.sum() * p.dt >= p.evidence_min_s))
                evd.append(float((u & ~dropped).sum() * p.dt >= p.evidence_min_s))
                ovl.append(float((inc & ~whole & ~dropped).sum() / inc.sum()) if inc.sum() else np.nan)
            rows.append(dict(base, metric="drop_evidence_sealframe", variant=dr, value=float(np.mean(evk))))
            rows.append(dict(base, metric="drop_evidence_discard", variant=dr, value=float(np.mean(evd))))
            rows.append(dict(base, metric="drop_owner_view", variant=dr, value=float(np.nanmean(ovl)) if inc.sum() else np.nan))
        new = not os.path.exists(OUT)
        with open(OUT, "a", newline="") as fh:
            wr = csv.DictWriter(fh, fieldnames=list(rows[0].keys()))
            if new: wr.writeheader()
            wr.writerows(rows)
        n += 1
print("DONE", n, "in", round(time.time() - t0), "s")


In [ ]:
%%writefile summarize_batch3.py
"""Summarise batch 3 (Table VIII). Bootstrap seed 7, 2,000 resamples."""
import pandas as pd, numpy as np, json
d = pd.read_csv("results/batch3_rows.csv"); rng = np.random.default_rng(7)
def ci(x):
    x = np.asarray(x, float); x = x[~np.isnan(x)]; bs = [x[rng.integers(0, len(x), len(x))].mean() for _ in range(2000)]
    return [round(float(x.mean()), 3), round(float(np.percentile(bs, 2.5)), 3), round(float(np.percentile(bs, 97.5)), 3), len(x)]
ov = d[d.metric == "owner_view"]
out = {"owner_view": {v: ci(g.value) for v, g in ov.groupby("variant")},
       "owner_view_by_time": {t: ci(ov[(ov.variant == "seal_whole_person") & ov.condition.str.startswith(t)].value) for t in ["day", "dusk", "night"]}}
for m in ["swap_collateral", "swap_evidence", "drop_evidence_sealframe", "drop_evidence_discard", "drop_owner_view"]:
    out[m] = {str(v): ci(g.value) for v, g in d[d.metric == m].groupby("variant")}
json.dump(out, open("results/batch3_summary.json", "w"), indent=1); print("written results/batch3_summary.json")


In [ ]:
%%writefile verify_paper.py
"""Compare every number reported in the manuscript (draft v0.1, sim v3) with the values recomputed in results/.
A value matches when the recomputed number, rounded to the same decimals as printed in the paper, equals it.
Usage: python verify_paper.py   (after run_all.sh, or on the shipped results/ files)."""
import json, sys
import pandas as pd

R = "results/"
S = json.load(open(R + "summary.json")); B2 = json.load(open(R + "batch2_summary.json")); B3 = json.load(open(R + "batch3_summary.json"))
ov = {o["design"]: o for o in S["overall"]}
T = {t["test"][:2]: t for t in S["primary_tests"]}
J = pd.read_csv(R + "sweep_joint_episodes.csv")
jm = J[J.condition == "day_clear"].groupby(["person_det_c50_pxm", "design"]).aie_face_s.mean()

# (section, label, paper value as printed, recomputed value)
REF = [
 ("Abstract/VII-A Tab.IV", "B1 raw AIE", "59.5", ov["B1_event_raw"]["aie"][0]),
 ("Tab.IV", "B1 AIE CI low", "49.5", ov["B1_event_raw"]["aie"][1]),
 ("Tab.IV", "B1 AIE CI high", "69.6", ov["B1_event_raw"]["aie"][2]),
 ("Tab.IV", "B1 face exposure", "10.5", ov["B1_event_raw"]["aie_face"][0]),
 ("Tab.IV", "B0 continuous AIE", "178.1", ov["B0_continuous_raw"]["aie"][0]),
 ("Tab.IV", "B2 blur AIE", "0.20", ov["B2_event_blur"]["aie"][0]),
 ("Tab.IV", "B2fc AIE", "0.00", ov["B2fc_event_blur_fc"]["aie"][0]),
 ("Tab.IV", "P AIE", "0.00", ov["P_proposed"]["aie"][0]),
 ("Tab.IV", "P evidence", "0.52", ov["P_proposed"]["evidence"][0]),
 ("Tab.IV", "B2fc evidence", "0.00", ov["B2fc_event_blur_fc"]["evidence"][0]),
 ("Tab.IV", "A1 AIE", "0.20", ov["A1_no_failclosed"]["aie"][0]),
 ("Tab.IV", "A3 collateral", "1.08", ov["A3_single_key"]["collateral"][0]),
 ("Tab.V", "T1 mean diff", "-59.5", T["T1"]["mean_diff"]),
 ("Tab.V", "T1 CI low", "-70.0", T["T1"]["ci"][0]),
 ("Tab.V", "T1 CI high", "-50.4", T["T1"]["ci"][1]),
 ("Tab.V", "T2 evidence diff", "0.52", T["T2"]["diff"]),
 ("Tab.V", "T2 discordant", "418", T["T2"]["n10"]),
 ("Tab.V", "T5 A1-P", "0.20", T["T5"]["mean_diff"]),
 ("Tab.V", "T6 A3-P", "1.08", T["T6"]["mean_diff"]),
 ("VII-A cluster", "T1 cluster CI low", "-64.0", B2["cluster"]["T1_AIE_P_minus_B1"]["ci"][0]),
 ("VII-A cluster", "T1 cluster CI high", "-55.4", B2["cluster"]["T1_AIE_P_minus_B1"]["ci"][1]),
 ("VII-A cluster", "T2 cluster CI low", "0.45", B2["cluster"]["T2_Ev_P_minus_B2fc"]["ci"][0]),
 ("VII-A cluster", "T6 cluster CI high", "1.30", B2["cluster"]["T6_Coll_A3_minus_P"]["ci"][1]),
] + [
 ("VII-B Fig.3", f"evidence {c}", v, S["by_condition"][f"{c}|P_proposed"]["evidence"][0])
 for c, v in [("day_clear", "1.00"), ("day_fog", "0.93"), ("day_rain", "0.67"), ("dusk_clear", "0.67"), ("dusk_fog", "0.62"),
              ("dusk_rain", "0.62"), ("night_clear", "0.13"), ("night_rain", "0.00"), ("night_fog", "0.00")]
] + [
 ("VII-B layout", "plate exposure, facing row", "98.0", S["by_layout"]["opposite_row|B1_event_raw"]["aie_plate"][0]),
 ("VII-C Fig.4", "B1 face, 2/min", "3.4", S["by_density"]["2.0|B1_event_raw"]["aie_face"][0]),
 ("VII-C Fig.4", "B1 face, 6/min", "7.4", S["by_density"]["6.0|B1_event_raw"]["aie_face"][0]),
 ("VII-C Fig.4", "B1 face, 15/min", "20.7", S["by_density"]["15.0|B1_event_raw"]["aie_face"][0]),
 ("VII-C Fig.4", "A3 collateral, 2/min", "0.44", S["by_density"]["2.0|A3_single_key"]["collateral"][0]),
 ("VII-C Fig.4", "A3 collateral, 15/min", "2.09", S["by_density"]["15.0|A3_single_key"]["collateral"][0]),
 ("VII-D Fig.5", "A1 face, c50=320, day", "6.6", S["sweep"]["day_clear|320.0|0.6|A1_no_failclosed"]["aie_face"][0]),
 ("VII-D Fig.5", "raw face, day sweep", "15.5", S["sweep"]["day_clear|320.0|0.6|B1_event_raw"]["aie_face"][0]),
 ("VII-D Tab.VI", "P, person c50=250", "0.05", jm[(250.0, "P_proposed")]),
 ("VII-D Tab.VI", "P, person c50=400", "5.50", jm[(400.0, "P_proposed")]),
 ("VII-D Tab.VI", "B1 raw", "16.4", jm[(400.0, "B1_event_raw")]),
 ("VII-D Tab.VI", "A1", "6.6", jm[(400.0, "A1_no_failclosed")]),
 ("VII-F Tab.VIII", "evidence tau=0.5", "0.59", B2["evidence_threshold"]["0.5"]["P_proposed"]),
 ("VII-F Tab.VIII", "evidence tau=3", "0.31", B2["evidence_threshold"]["3.0"]["P_proposed"]),
 ("VII-F Tab.VIII", "raw AIE clip 20/40", "156.7", B2["trigger"]["20/40/1.0"]["B1_aie"]),
 ("VII-F Tab.VIII", "raw AIE trigger 0.5 m", "7.5", B2["trigger"]["10/20/0.5"]["B1_aie"]),
 ("VII-F Tab.VIII", "evidence trigger 0.5 m", "0.17", B2["trigger"]["10/20/0.5"]["P_evidence"]),
 ("VII-F Tab.VIII", "evidence night q=0.65", "0.67", B2["condition_factor"]["night_q=0.65"]["P_evidence"]),
 ("VII-F Tab.VIII", "raw AIE rain q=0.9", "499.7", B2["condition_factor"]["rain_q=0.9"]["B1_aie"]),
 ("IV/VII-F", "A1 violation rate (%)", "1.2", 100 * B2["A1"]["overall"]),
 ("VII-G Tab.IX", "owner view, whole person", "0.21", B3["owner_view"]["seal_whole_person"][0]),
 ("VII-G Tab.IX", "owner view, night", "0.13", B3["owner_view_by_time"]["night"][0]),
 ("VII-G Tab.IX", "swap collateral p=0.2", "0.03", B3["swap_collateral"]["0.2"][0]),
 ("VII-G Tab.IX", "evidence, 50% frames discarded", "0.41", B3["drop_evidence_discard"]["0.5"][0]),
]

def fmt_like(ref, x):
    d = len(ref.split(".")[1]) if "." in ref else 0
    return f"{x:.{d}f}"

bad = 0
print(f"{'section':16s} {'quantity':34s} {'paper':>8s} {'recomputed':>11s}  status")
for sec, lab, ref, val in REF:
    got = fmt_like(ref, float(val))
    ok = got.lstrip("-") == ref.lstrip("-") and (got.startswith("-") == ref.startswith("-") or float(got) == 0)
    bad += (not ok)
    print(f"{sec:16s} {lab:34s} {ref:>8s} {got:>11s}  {'OK' if ok else 'MISMATCH'}")
print(f"\n{len(REF) - bad}/{len(REF)} paper values reproduced.")
print("Not reproducible from code: Table VII (real-image check needs the author's photographs, which are not released).")
sys.exit(1 if bad else 0)


## 1. Sanity checks (Section VI)

In [ ]:
!python checks.py

## 2. QUICK: six episodes against published per-episode values (Section VII-A data)

In [ ]:
import json, numpy as np, sim
EXPECTED = json.loads('''[{"condition": "day_clear", "dens": 6.0, "layout": "opposite_row", "seed": 0, "design": "B0_continuous_raw", "aie_face_s": 31.5, "aie_plate_s": 1739.5, "evidence": 1.0, "collateral_s": 0.0}, {"condition": "day_clear", "dens": 6.0, "layout": "opposite_row", "seed": 0, "design": "B1_event_raw", "aie_face_s": 16.5, "aie_plate_s": 451.5, "evidence": 1.0, "collateral_s": 0.0}, {"condition": "day_clear", "dens": 6.0, "layout": "opposite_row", "seed": 0, "design": "B2_event_blur", "aie_face_s": 0.0, "aie_plate_s": 0.0, "evidence": 0.0, "collateral_s": 0.0}, {"condition": "day_clear", "dens": 6.0, "layout": "opposite_row", "seed": 0, "design": "B2fc_event_blur_fc", "aie_face_s": 0.0, "aie_plate_s": 0.0, "evidence": 0.0, "collateral_s": 0.0}, {"condition": "day_clear", "dens": 6.0, "layout": "opposite_row", "seed": 0, "design": "P_proposed", "aie_face_s": 0.0, "aie_plate_s": 0.0, "evidence": 1.0, "collateral_s": 0.0}, {"condition": "day_clear", "dens": 6.0, "layout": "opposite_row", "seed": 0, "design": "A1_no_failclosed", "aie_face_s": 0.0, "aie_plate_s": 0.0, "evidence": 1.0, "collateral_s": 0.0}, {"condition": "day_clear", "dens": 6.0, "layout": "opposite_row", "seed": 0, "design": "A2_continuous_sealed", "aie_face_s": 0.0, "aie_plate_s": 0.0, "evidence": 1.0, "collateral_s": 0.0}, {"condition": "day_clear", "dens": 6.0, "layout": "opposite_row", "seed": 0, "design": "A3_single_key", "aie_face_s": 0.0, "aie_plate_s": 0.0, "evidence": 1.0, "collateral_s": 1.0}, {"condition": "night_clear", "dens": 15.0, "layout": "no_opposite", "seed": 1, "design": "B0_continuous_raw", "aie_face_s": 5.0, "aie_plate_s": 0.0, "evidence": 1.0, "collateral_s": 0.0}, {"condition": "night_clear", "dens": 15.0, "layout": "no_opposite", "seed": 1, "design": "B1_event_raw", "aie_face_s": 5.0, "aie_plate_s": 0.0, "evidence": 1.0, "collateral_s": 0.0}, {"condition": "night_clear", "dens": 15.0, "layout": "no_opposite", "seed": 1, "design": "B2_event_blur", "aie_face_s": 0.0, "aie_plate_s": 0.0, "evidence": 0.0, "collateral_s": 0.0}, {"condition": "night_clear", "dens": 15.0, "layout": "no_opposite", "seed": 1, "design": "B2fc_event_blur_fc", "aie_face_s": 0.0, "aie_plate_s": 0.0, "evidence": 0.0, "collateral_s": 0.0}, {"condition": "night_clear", "dens": 15.0, "layout": "no_opposite", "seed": 1, "design": "P_proposed", "aie_face_s": 0.0, "aie_plate_s": 0.0, "evidence": 1.0, "collateral_s": 0.0}, {"condition": "night_clear", "dens": 15.0, "layout": "no_opposite", "seed": 1, "design": "A1_no_failclosed", "aie_face_s": 0.0, "aie_plate_s": 0.0, "evidence": 1.0, "collateral_s": 0.0}, {"condition": "night_clear", "dens": 15.0, "layout": "no_opposite", "seed": 1, "design": "A2_continuous_sealed", "aie_face_s": 0.0, "aie_plate_s": 0.0, "evidence": 1.0, "collateral_s": 0.0}, {"condition": "night_clear", "dens": 15.0, "layout": "no_opposite", "seed": 1, "design": "A3_single_key", "aie_face_s": 0.0, "aie_plate_s": 0.0, "evidence": 1.0, "collateral_s": 1.0}, {"condition": "dusk_fog", "dens": 2.0, "layout": "opposite_row", "seed": 2, "design": "B0_continuous_raw", "aie_face_s": 6.5, "aie_plate_s": 0.0, "evidence": 0.0, "collateral_s": 0.0}, {"condition": "dusk_fog", "dens": 2.0, "layout": "opposite_row", "seed": 2, "design": "B1_event_raw", "aie_face_s": 2.5, "aie_plate_s": 0.0, "evidence": 0.0, "collateral_s": 0.0}, {"condition": "dusk_fog", "dens": 2.0, "layout": "opposite_row", "seed": 2, "design": "B2_event_blur", "aie_face_s": 0.0, "aie_plate_s": 0.0, "evidence": 0.0, "collateral_s": 0.0}, {"condition": "dusk_fog", "dens": 2.0, "layout": "opposite_row", "seed": 2, "design": "B2fc_event_blur_fc", "aie_face_s": 0.0, "aie_plate_s": 0.0, "evidence": 0.0, "collateral_s": 0.0}, {"condition": "dusk_fog", "dens": 2.0, "layout": "opposite_row", "seed": 2, "design": "P_proposed", "aie_face_s": 0.0, "aie_plate_s": 0.0, "evidence": 0.0, "collateral_s": 0.0}, {"condition": "dusk_fog", "dens": 2.0, "layout": "opposite_row", "seed": 2, "design": "A1_no_failclosed", "aie_face_s": 0.0, "aie_plate_s": 0.0, "evidence": 0.0, "collateral_s": 0.0}, {"condition": "dusk_fog", "dens": 2.0, "layout": "opposite_row", "seed": 2, "design": "A2_continuous_sealed", "aie_face_s": 0.0, "aie_plate_s": 0.0, "evidence": 0.0, "collateral_s": 0.0}, {"condition": "dusk_fog", "dens": 2.0, "layout": "opposite_row", "seed": 2, "design": "A3_single_key", "aie_face_s": 0.0, "aie_plate_s": 0.0, "evidence": 0.0, "collateral_s": 0.0}, {"condition": "day_rain", "dens": 15.0, "layout": "opposite_row", "seed": 3, "design": "B0_continuous_raw", "aie_face_s": 54.5, "aie_plate_s": 565.0, "evidence": 1.0, "collateral_s": 0.0}, {"condition": "day_rain", "dens": 15.0, "layout": "opposite_row", "seed": 3, "design": "B1_event_raw", "aie_face_s": 39.5, "aie_plate_s": 355.0, "evidence": 1.0, "collateral_s": 0.0}, {"condition": "day_rain", "dens": 15.0, "layout": "opposite_row", "seed": 3, "design": "B2_event_blur", "aie_face_s": 0.5, "aie_plate_s": 1.5, "evidence": 0.0, "collateral_s": 0.0}, {"condition": "day_rain", "dens": 15.0, "layout": "opposite_row", "seed": 3, "design": "B2fc_event_blur_fc", "aie_face_s": 0.0, "aie_plate_s": 0.0, "evidence": 0.0, "collateral_s": 0.0}, {"condition": "day_rain", "dens": 15.0, "layout": "opposite_row", "seed": 3, "design": "P_proposed", "aie_face_s": 0.0, "aie_plate_s": 0.0, "evidence": 1.0, "collateral_s": 0.0}, {"condition": "day_rain", "dens": 15.0, "layout": "opposite_row", "seed": 3, "design": "A1_no_failclosed", "aie_face_s": 0.5, "aie_plate_s": 1.5, "evidence": 1.0, "collateral_s": 0.0}, {"condition": "day_rain", "dens": 15.0, "layout": "opposite_row", "seed": 3, "design": "A2_continuous_sealed", "aie_face_s": 0.0, "aie_plate_s": 0.0, "evidence": 1.0, "collateral_s": 0.0}, {"condition": "day_rain", "dens": 15.0, "layout": "opposite_row", "seed": 3, "design": "A3_single_key", "aie_face_s": 0.0, "aie_plate_s": 0.0, "evidence": 1.0, "collateral_s": 4.5}, {"condition": "night_fog", "dens": 6.0, "layout": "no_opposite", "seed": 4, "design": "B0_continuous_raw", "aie_face_s": 1.5, "aie_plate_s": 0.0, "evidence": 0.0, "collateral_s": 0.0}, {"condition": "night_fog", "dens": 6.0, "layout": "no_opposite", "seed": 4, "design": "B1_event_raw", "aie_face_s": 1.5, "aie_plate_s": 0.0, "evidence": 0.0, "collateral_s": 0.0}, {"condition": "night_fog", "dens": 6.0, "layout": "no_opposite", "seed": 4, "design": "B2_event_blur", "aie_face_s": 0.0, "aie_plate_s": 0.0, "evidence": 0.0, "collateral_s": 0.0}, {"condition": "night_fog", "dens": 6.0, "layout": "no_opposite", "seed": 4, "design": "B2fc_event_blur_fc", "aie_face_s": 0.0, "aie_plate_s": 0.0, "evidence": 0.0, "collateral_s": 0.0}, {"condition": "night_fog", "dens": 6.0, "layout": "no_opposite", "seed": 4, "design": "P_proposed", "aie_face_s": 0.0, "aie_plate_s": 0.0, "evidence": 0.0, "collateral_s": 0.0}, {"condition": "night_fog", "dens": 6.0, "layout": "no_opposite", "seed": 4, "design": "A1_no_failclosed", "aie_face_s": 0.0, "aie_plate_s": 0.0, "evidence": 0.0, "collateral_s": 0.0}, {"condition": "night_fog", "dens": 6.0, "layout": "no_opposite", "seed": 4, "design": "A2_continuous_sealed", "aie_face_s": 0.0, "aie_plate_s": 0.0, "evidence": 0.0, "collateral_s": 0.0}, {"condition": "night_fog", "dens": 6.0, "layout": "no_opposite", "seed": 4, "design": "A3_single_key", "aie_face_s": 0.0, "aie_plate_s": 0.0, "evidence": 0.0, "collateral_s": 0.0}, {"condition": "dusk_clear", "dens": 2.0, "layout": "no_opposite", "seed": 5, "design": "B0_continuous_raw", "aie_face_s": 11.0, "aie_plate_s": 0.0, "evidence": 0.0, "collateral_s": 0.0}, {"condition": "dusk_clear", "dens": 2.0, "layout": "no_opposite", "seed": 5, "design": "B1_event_raw", "aie_face_s": 2.0, "aie_plate_s": 0.0, "evidence": 0.0, "collateral_s": 0.0}, {"condition": "dusk_clear", "dens": 2.0, "layout": "no_opposite", "seed": 5, "design": "B2_event_blur", "aie_face_s": 0.0, "aie_plate_s": 0.0, "evidence": 0.0, "collateral_s": 0.0}, {"condition": "dusk_clear", "dens": 2.0, "layout": "no_opposite", "seed": 5, "design": "B2fc_event_blur_fc", "aie_face_s": 0.0, "aie_plate_s": 0.0, "evidence": 0.0, "collateral_s": 0.0}, {"condition": "dusk_clear", "dens": 2.0, "layout": "no_opposite", "seed": 5, "design": "P_proposed", "aie_face_s": 0.0, "aie_plate_s": 0.0, "evidence": 0.0, "collateral_s": 0.0}, {"condition": "dusk_clear", "dens": 2.0, "layout": "no_opposite", "seed": 5, "design": "A1_no_failclosed", "aie_face_s": 0.0, "aie_plate_s": 0.0, "evidence": 0.0, "collateral_s": 0.0}, {"condition": "dusk_clear", "dens": 2.0, "layout": "no_opposite", "seed": 5, "design": "A2_continuous_sealed", "aie_face_s": 0.0, "aie_plate_s": 0.0, "evidence": 0.0, "collateral_s": 0.0}, {"condition": "dusk_clear", "dens": 2.0, "layout": "no_opposite", "seed": 5, "design": "A3_single_key", "aie_face_s": 0.0, "aie_plate_s": 0.0, "evidence": 0.0, "collateral_s": 1.0}]''')
bad = 0
for c, d, l, s in sorted({(e['condition'], e['dens'], e['layout'], e['seed']) for e in EXPECTED}):
    _, _, res = sim.run_episode(sim.Params(condition=c, ped_rate_per_min=d, layout=l, seed=1000 * s + 7))
    got = {r['design']: r for r in res}
    for e in [e for e in EXPECTED if (e['condition'], e['dens'], e['layout'], e['seed']) == (c, d, l, s)]:
        r = got[e['design']]
        same = all(abs(r[k] - e[k]) < 1e-6 for k in ('aie_face_s', 'aie_plate_s', 'collateral_s')) and r['evidence'] == e['evidence']
        bad += not same
    print(f'{c:12s} {d:5.1f}/min {l:13s} seed {s}:', 'all 8 designs match' if bad == 0 else f'{bad} mismatches so far')
print('QUICK result:', 'PASS' if bad == 0 else f'FAIL ({bad} mismatches)')

## 3. FULL: rerun everything and compare every paper value

In [ ]:
if MODE == "FULL":
    !python bench.py main && python bench.py sweep && python bench.py sweep_person
    !python joint_stress.py && python analyze.py
    !python batch2.py && python analyze_batch2.py && python batch3.py && python summarize_batch3.py
    !python verify_paper.py
else:
    print('Set MODE = "FULL" to rerun all experiments (about 1.5 h).')